In [1]:
import os
import json
import re
from pathlib import Path
from collections import defaultdict

import torch
from datasets import Dataset

from rapidfireai import Experiment
from rapidfireai.automl import List, RFGridSearch, RFModelConfig, RFLoraConfig, RFSFTConfig

# RapidFire AI Tutorial Use Case: SFT for Customer Support Q&A Chatbot

## Load the data and Specify Train and Eval Partitions

In [ ]:
PROJECT_DIR = Path(".")  
SCHEMAS_DIR = PROJECT_DIR / "schemas" 
TRAIN_PATH = PROJECT_DIR / "train.json"  
VAL_PATH = PROJECT_DIR / "validation.json" 

SYSTEM_PROMPT = """You are a schema linking model for text-to-SQL.
Given a natural language question and a database schema, return ONLY valid JSON.
The JSON format must be:
{"TableName": ["Column1", "Column2"], "AnotherTable": []}

Rules:
- Use only table and column names from the provided schema.
- Include a table with [] if the table is referenced but no specific column is needed.
- Do not explain.
- Do not output markdown.
"""  

In [ ]:

# load schema
def schema_file_for_db(db_id, schemas_dir=SCHEMAS_DIR):
    fname = db_id.replace(" ", "_").replace("/", "_") + ".json"  
    return schemas_dir / fname  # return corresponding schema JSON path of a database


def load_spider_schema(db_id, schemas_dir=SCHEMAS_DIR):
    with open(schema_file_for_db(db_id, schemas_dir)) as f:  
        s = json.load(f)  # read schema JSON

    table_names = s["table_names_original"]  
    columns_by_table = {t: [] for t in table_names}  # initialize {table: [columns]} 

    for tidx, cname in s["column_names_original"]:  # 
        if tidx == -1:  # tidx=-1 :synthetic '*'
            continue
        columns_by_table[table_names[tidx]].append(cname)  

    return columns_by_table  # dictionary



# 1 serialize_schema
def serialize_schema(db_id):
    schema = load_spider_schema(db_id)  # load database schema
    lines = [f"Database: {db_id}", "Schema:"]  # 
    for table, cols in schema.items(): 
        lines.append(f"- {table}({', '.join(cols)})") 
    return "\n".join(lines)  

#construct user prompt
def build_user_prompt(question, db_id):
    return f"""{serialize_schema(db_id)}

Question:
{question}

Return schema_links JSON only."""  # prompt

In [ ]:
def load_records(path):
    with open(path) as f:
        return json.load(f)


train_records = load_records(TRAIN_PATH)
val_records = load_records(VAL_PATH)


def add_prompt_fields(records, train=True):
    out = []
    for row in records:
        new_row = dict(row)
        new_row["user_prompt"] = build_user_prompt(row["question"], row["db_id"])
        new_row["assistant_response"] = json.dumps(
            row["schema_links"],
            ensure_ascii=False,
            separators=(",", ":"),
        )

        out.append(new_row)

        if train:
            if row["db_id"].startswith("SBODemoUS"):
                out.append(new_row) # repeat f
            if len(row["schema_links"]) >= 2:
                out.append(new_row)  # repeat
    return out

# #  get train and validation dataset
train_dataset = Dataset.from_list(add_prompt_fields(train_records, train=True)).shuffle(seed=42)
eval_dataset = Dataset.from_list(add_prompt_fields(val_records, train=False)).shuffle(seed=42)

In [5]:
print('length of train dataset: ', len(train_dataset)) #487
print('length of eval dataset', len(eval_dataset)) #101

length of train dataset:  487
length of eval dataset 101


## Define Data Processing Function

In [ ]:
# 
def formatting_function(row):# #stiching the prompt, question and response
    system_prompt = """You are a schema linking model for text-to-SQL.
Given a natural language question and a database schema, return ONLY valid JSON.
The JSON format must be:
{"TableName": ["Column1", "Column2"], "AnotherTable": []}

Rules:
- Use only table and column names from the provided schema.
- Include a table with [] if the table is referenced but no specific column is needed.
- Do not explain.
- Do not output markdown.
"""


    return {
        "prompt": [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": row["user_prompt"]},
        ],
        "completion": [
            {"role": "assistant", "content": row["assistant_response"]}
        ],
    }


## Post preprocess

In [ ]:
def extract_json_object(text):
    if not isinstance(text, str):  # avoid json.loads error when prediction is not str
        return {}

    text = text.strip()  # remove spaces
    text = re.sub(r"^```(?:json)?", "", text).strip()  # remove markdown JSON 
    text = re.sub(r"```$", "", text).strip()  # remove markdown 

    try:
        obj = json.loads(text)  # parse to JSON
        return obj if isinstance(obj, dict) else {}  
    except Exception:
        pass  # if error, pass

    match = re.search(r"\{.*\}", text, flags=re.S)  # extract {...}
    if not match:
        return {}

    try:
        obj = json.loads(match.group(0))  # extract JSON 
        return obj if isinstance(obj, dict) else {}  # retain JSON only
    except Exception:
        return {}  # 

#post process
# def canonicalize_schema_links(raw_links, db_id):#把模型预测出的 schema links 清洗（clean）、纠正大小写、删掉不存在的 table/column，最后输出合法格式。
#     schema = load_spider_schema(db_id)  # 加载当前 db_id 的合法表列集合
#     table_map = {t.lower(): t for t in schema}  # 表名大小写归一化，输出时恢复原始 casing
#     col_maps = {
#         t: {c.lower(): c for c in cols}  # 每张表内列名大小写归一化，输出时恢复原始 casing
#         for t, cols in schema.items()
#     }

#     clean = {}  # 保存过滤后的合法 schema_links
#     if not isinstance(raw_links, dict):  # 如果模型输出不是 dict，直接返回空结果
#         return clean

#     for raw_t, raw_cols in raw_links.items():  # 遍历模型预测的表和列
#         t_key = str(raw_t).lower()  # 表名转小写用于匹配 schema
#         if t_key not in table_map:  # 过滤 hallucinated table，避免 precision 被扣分
#             continue

#         table = table_map[t_key]  # 恢复 schema 中的原始表名 casing
#         cols = []  # 当前表下保留下来的合法列
#         if isinstance(raw_cols, list):  # 只有 list 才作为列集合处理
#             for c in raw_cols:
#                 c_key = str(c).lower()  # 列名转小写用于匹配 schema
#                 if c_key in col_maps[table]:  # 只保留属于该表的合法列
#                     cols.append(col_maps[table][c_key])  # 恢复 schema 中的原始列名 casing

#         clean[table] = sorted(set(cols))  # 去重并排序，保证输出稳定；空列表表示 table-only link

#     return clean  # 返回可直接提交给 eval.py 的过滤后结果

## Initialize Experiment

In [ ]:
experiment = Experiment(
    experiment_name="schema-linking-baseline-qwen-lora",  # RapidFireAI experiment name
    mode="fit",  # fit 
)

The previously running experiment schema-linking-baseline-qwen-lora_12 was forcibly ended. Created a new experiment 'schema-linking-baseline-qwen-lora_13' with Experiment ID: 15 and Metric Experiment ID: 87 at /home/duw003/rapidfireai/rapidfire_experiments/schema-linking-baseline-qwen-lora_13


## Define Custom Eval Metrics Function

In [ ]:
import numpy as np
from eval import prf
def links_to_pairs(d):# JSON schema links transfer to (table,column) 
    pairs = set()
    if not isinstance(d, dict):
        return pairs

    for table, cols in d.items():
        if isinstance(cols, list):
            for col in cols:
                pairs.add((str(table), str(col)))
        else:
            pairs.add((str(table), None))

    return pairs


def schema_link_metrics(eval_preds):
    predictions, labels = eval_preds  # generate predictions

    valid_json = 0  # 
    exact = 0  
    total = len(predictions)  
    ps, rs, f1s = [], [], []

    for pred, label in zip(predictions, labels):
        pred_json = extract_json_object(pred)# extract JSON
        gold_json = extract_json_object(label)#

        if pred_json:
            valid_json += 1
        if pred_json == gold_json:
            exact += 1

        pred_pairs = links_to_pairs(pred_json)
        gold_pairs = links_to_pairs(gold_json)

        p, r, f1 = prf(pred_pairs, gold_pairs)

        ps.append(p)
        rs.append(r)
        f1s.append(f1)

    avg_p = np.mean(ps)
    avg_r = np.mean(rs)
    avg_f1 = np.mean(f1s)

    return {
        "valid_json_rate": round(valid_json / max(total, 1), 4),  # JSON legal rate
        "exact_json_rate": round(exact / max(total, 1), 4),  # exact matching rate
        "pair_precision": round(avg_p, 4),
        "pair_recall": round(avg_r, 4),
        "pair_f1": round(avg_f1, 4),
    }


## Define Multi-Config Knobs for Model, LoRA, and SFT Trainer using RapidFire AI Wrapper APIs

In [ ]:
peft_configs = List([  
    # RFLoraConfig(
    #     r=8,  # small
    #     lora_alpha=16,  # 
    #     lora_dropout=0.05,  # 轻
    #     target_modules=["q_proj", "v_proj"],  # 
    #     bias="none",  # do not train bais
    # ),
    RFLoraConfig(
        r=16,  # larger LoRA rank
        lora_alpha=32,  # 
        lora_dropout=0.05,  
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],  
        bias="none",  
    ),
])

common_generation_config = {  
    "max_new_tokens": 256,  
    "do_sample": False,  
    "temperature": None, 
    "top_p": None, 
    "repetition_penalty": 1.02, 
}

common_model_kwargs = {  
    "device_map": "auto",  
    "torch_dtype": "auto", 
    "use_cache": False,  
}

In [ ]:
# 2 base models x 2 learning rates x 2 LoRA configs = 8 configs，
config_set = List([  
    # RFModelConfig(
    #     model_name="Qwen/Qwen2.5-0.5B-Instruct",  # 0.5B small model
    #     peft_config=peft_configs,  
    #     training_args=RFSFTConfig( 
    #         learning_rate=2e-4, 
    #         lr_scheduler_type="linear", 
    #         per_device_train_batch_size=4,  
    #         per_device_eval_batch_size=4,  
    #         gradient_accumulation_steps=8,  
    #         num_train_epochs=4, 
    #         logging_steps=5,  
    #         eval_strategy="steps",  
    #         eval_steps=25,  
    #         save_steps=25, 
    #         bf16=True, 
    #     ),
    #     model_type="causal_lm", 
    #     model_kwargs=common_model_kwargs, 
    #     formatting_func=formatting_function, 
    #     compute_metrics=schema_link_metrics, 
    #     generation_config=common_generation_config,  
    # ),
    # RFModelConfig(
    #     model_name="Qwen/Qwen2.5-0.5B-Instruct",  
    #     peft_config=peft_configs,  
    #     training_args=RFSFTConfig(
    #         learning_rate=5e-4,  
    #         lr_scheduler_type="cosine",  
    #         per_device_train_batch_size=1,  
    #         per_device_eval_batch_size=1,  
    #         gradient_accumulation_steps=8, 
    #         num_train_epochs=4, 
    #         logging_steps=5, 
    #         eval_strategy="steps", 
    #         eval_steps=25, 
    #         save_steps=25, 
    #         bf16=True, 
    #     ),
    #     model_type="causal_lm",  
    #     model_kwargs=common_model_kwargs, 
    #     formatting_func=formatting_function, 
    #     compute_metrics=schema_link_metrics, 
    #     generation_config=common_generation_config, 
    # ),
    # RFModelConfig(
    #     model_name="Qwen/Qwen2.5-1.5B-Instruct",  # 1.5B larger model
    #     peft_config=peft_configs, 
    #     training_args=RFSFTConfig(
    #         learning_rate=1e-4, 
    #         lr_scheduler_type="cosine",  
    #         per_device_train_batch_size=1, 
    #         per_device_eval_batch_size=1, 
    #         gradient_accumulation_steps=8, 
    #         num_train_epochs=3, 
    #         logging_steps=5, 
    #         eval_strategy="steps", 
    #         eval_steps=50, 
    #         save_steps=100, 
    #         bf16=True, 
    #     ),
    #     model_type="causal_lm", 
    #     model_kwargs=common_model_kwargs, 
    #     formatting_func=formatting_function, 
    #     compute_metrics=schema_link_metrics, 
    #     generation_config=common_generation_config, 
    # ),
    # RFModelConfig(
    #     model_name="Qwen/Qwen2.5-1.5B-Instruct", 
    #     peft_config=peft_configs, 
    #     training_args=RFSFTConfig(
    #         learning_rate=2e-4,  
    #         lr_scheduler_type="cosine",  
    #         per_device_train_batch_size=1, 
    #         per_device_eval_batch_size=1, 
    #         gradient_accumulation_steps=8,
    #         num_train_epochs=3,
    #         logging_steps=5, 
    #         eval_strategy="steps",  
    #         eval_steps=25,  
    #         save_steps=25, 
    #         bf16=True,  
    #     ),
    #     model_type="causal_lm", 
    #     model_kwargs=common_model_kwargs,  
    #     formatting_func=formatting_function, 
    #     compute_metrics=schema_link_metrics, 
    #     generation_config=common_generation_config,  
    # ),
    # RFModelConfig(
    #     model_name="Qwen/Qwen3-1.7B",  
    #     peft_config=peft_configs,  
    #     training_args=RFSFTConfig(
    #         learning_rate=1e-4,  
    #         lr_scheduler_type="cosine",  
    #         per_device_train_batch_size=1,  
    #         per_device_eval_batch_size=1,  
    #         gradient_accumulation_steps=8, 
    #         num_train_epochs=3,  
    #         logging_steps=5,  
    #         eval_strategy="steps",  
    #         eval_steps=50,  
    #         save_steps=100,  
    #         bf16=True, 
    #     ),
    #     model_type="causal_lm", 
    #     model_kwargs=common_model_kwargs, 
    #     formatting_func=formatting_function, 
    #     compute_metrics=schema_link_metrics,  
    #     generation_config=common_generation_config, 
    # ),
    RFModelConfig(
        model_name="Qwen/Qwen3-1.7B",  
        peft_config=peft_configs, 
        training_args=RFSFTConfig(
            learning_rate=3e-4,  
            lr_scheduler_type="cosine",  
            per_device_train_batch_size=1,  
            per_device_eval_batch_size=1, 
            gradient_accumulation_steps=8,  
            num_train_epochs=3, 
            logging_steps=5,  
            eval_strategy="steps", 
            eval_steps=50, 
            save_steps=100,  
            bf16=True, 
        ),
        model_type="causal_lm",  
        model_kwargs=common_model_kwargs, 
        formatting_func=formatting_function,  
        compute_metrics=schema_link_metrics, 
        generation_config=common_generation_config, 
    ),
    # RFModelConfig(
    #     model_name="Qwen/Qwen3-1.7B", 
    #     peft_config=peft_configs, 
    #     training_args=RFSFTConfig(
    #         learning_rate=5e-4,  
    #         lr_scheduler_type="cosine", 
    #         per_device_train_batch_size=1, 
    #         per_device_eval_batch_size=1, 
    #         gradient_accumulation_steps=8,  
    #         num_train_epochs=3, 
    #         logging_steps=5,
    #         eval_strategy="steps", 
    #         eval_steps=100,
    #         save_steps=150,
    #         bf16=True, 
    #     ),
    #     model_type="causal_lm", 
    #     model_kwargs=common_model_kwargs, 
    #     formatting_func=formatting_function,  
    #     compute_metrics=schema_link_metrics, 
    #     generation_config=common_generation_config, 
    # ),
])

## Define Model Creation Function for All Model Types Across Configs

In [ ]:
def create_model(model_config): #
     """Function to create model object for any given config; must return tuple of (model, tokenizer)"""
     from transformers import AutoModelForCausalLM, AutoTokenizer, AutoModelForSeq2SeqLM, AutoModelForMaskedLM

     model_name = model_config["model_name"]
     model_type = model_config["model_type"]
     model_kwargs = model_config["model_kwargs"]
     #create or load the model
 
     if model_type == "causal_lm":
          model = AutoModelForCausalLM.from_pretrained(model_name, **model_kwargs)
     elif model_type == "seq2seq_lm":
          model = AutoModelForSeq2SeqLM.from_pretrained(model_name, **model_kwargs)
     elif model_type == "masked_lm":
          model = AutoModelForMaskedLM.from_pretrained(model_name, **model_kwargs)
     elif model_type == "custom":
          # Handle custom model loading logic, e.g., loading your own checkpoints
          # model = ... 
          pass
     else:
          # Default to causal LM
          model = AutoModelForCausalLM.from_pretrained(model_name, **model_kwargs)
      
     tokenizer = AutoTokenizer.from_pretrained(model_name)
      
     return (model,tokenizer)# return model and tokenizer from hugging face


## Generate Config Group

In [ ]:
config_group = RFGridSearch(
    configs=config_set,  # config
    trainer_type="SFT",  # 
)

## Run Multi-Config Training

In [ ]:
experiment.run_fit(
    config_group,  # 
    create_model,  # 
    train_dataset,  # train Dataset
    eval_dataset,  # val Dataset，
    num_chunks=4,  # 
    seed=42, 
)

INFO 05-29 09:15:00 [__init__.py:216] Automatically detected platform cuda.
Started 1 worker processes successfully
Created workers
INFO 05-29 09:15:15 [__init__.py:216] Automatically detected platform cuda.


## End Current Experiment

In [ ]:
experiment.end() 

Experiment schema-linking-baseline-qwen-lora_13 ended
Workers stopped
